In [ ]:
print("Configure figure inputs")

TF_NAMES = ["GATA1", "MAX", "ABF1", "CTCF"]
HOCOMOCO_COLLECTION = "H14CORE"
EXACT_MAX_SEQUENCES = 5_000_000
EPSILON_LIMITS = (1e-4, 0.45)
FIT_EPSILON_RANGE = (1e-3, 0.20)
N_SADDLE_POINTS = 600
N_EXACT_BINS = 90
N_TOUZET_BINS = 80
TOUZET_KW = dict(tol=0.5, max_time=15, max_dict_size=10_000_000)
CRP_LOGO_TEMPERATURE = 1.0
ABF1_LOGO_TEMPERATURE = 0.2
SAVE_FIGURE = True

In [ ]:
print("Load libraries and figure style")

from pathlib import Path
import json
import ssl
import sys
import urllib.request

import logomaker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.optimize import curve_fit

sys.path.insert(0, str(Path('../../src').resolve()))
from landscape import Landscape
from touzet import tail_count

if Path("figs/figS3").exists():
    FIG_DIR = Path("figs/figS3")
    FIGS_DIR = FIG_DIR.parent
elif Path("../style_config.py").exists():
    FIG_DIR = Path(".")
    FIGS_DIR = Path("..")
else:
    FIG_DIR = Path(".")
    FIGS_DIR = Path(".")

DATA_DIR = FIGS_DIR / "data"
HOCOMOCO_DIR = DATA_DIR / "hocomoco"
TOUZET_CACHE_DIR = FIG_DIR / "touzet_cache"
HOCOMOCO_DIR.mkdir(parents=True, exist_ok=True)
TOUZET_CACHE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(FIGS_DIR.resolve()))
import style_config as sc

BASES = ["A", "C", "G", "T"]
QUALITY_RANK = {"A": 0, "B": 1, "C": 2, "D": 3}

In [ ]:
print("Define motif loading functions")

def softmax_rows(x, temperature=1.0):
    z = np.asarray(x, dtype=float) / temperature
    z = z - z.max(axis=1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=1, keepdims=True)


def information_logo_matrix(p):
    p = np.asarray(p, dtype=float)
    p = p / p.sum(axis=1, keepdims=True)
    entropy_terms = np.zeros_like(p)
    positive = p > 0
    entropy_terms[positive] = p[positive] * np.log2(p[positive])
    information = np.log2(p.shape[1]) + entropy_terms.sum(axis=1)
    return p * information[:, None]


def standardized_weight_matrix(theta):
    theta = np.asarray(theta, dtype=float)
    L = theta.shape[0]
    centered = theta - theta.mean(axis=1, keepdims=True)
    scale = centered.std() * np.sqrt(L)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError("Cannot standardize a motif with zero additive variance")
    return centered / scale


def hocomoco_cache_file(collection):
    return HOCOMOCO_DIR / f"{collection}_annotation.jsonl"


def hocomoco_url(collection):
    return f"https://hocomoco14.autosome.org/final_bundle/hocomoco14/{collection}/{collection}_annotation.jsonl"


def download_hocomoco(collection):
    cache = hocomoco_cache_file(collection)
    if cache.exists():
        print(f"  Using cached {cache}")
        return cache
    print(f"  Downloading {collection} annotations")
    context = ssl._create_unverified_context()
    with urllib.request.urlopen(hocomoco_url(collection), context=context, timeout=60) as response:
        cache.write_bytes(response.read())
    return cache


def load_hocomoco_records(collection):
    cache = download_hocomoco(collection)
    records = []
    with cache.open() as handle:
        for line in handle:
            records.append(json.loads(line))
    return records


def hocomoco_sort_key(record):
    quality = QUALITY_RANK.get(record.get("quality", "Z"), 99)
    length = record.get("length", 999)
    distance = abs(length - 10)
    datatype = record.get("datatype") or ""
    information = record.get("information_content_per_base") or 0.0
    subtype = record.get("subtype_order", 99)
    return (quality, distance, -len(datatype), -information, subtype, record.get("name", ""))


def select_hocomoco_record(query, records):
    exact = [record for record in records if record.get("name") == query]
    if exact:
        return exact[0]
    hits = [record for record in records if record.get("tf", "").upper() == query.upper()]
    if not hits:
        raise KeyError(f"No HOCOMOCO motif found for {query}")
    return sorted(hits, key=hocomoco_sort_key)[0]


def record_from_hocomoco(query, records):
    record = select_hocomoco_record(query, records)
    theta = np.array(record["pwm"], dtype=float)
    pfm = np.array(record["pfm"], dtype=float)
    theta = standardized_weight_matrix(theta)
    return dict(
        query=query,
        tf=record["tf"],
        motif_id=record["name"],
        source=f"HOCOMOCO {HOCOMOCO_COLLECTION}",
        theta=theta,
        pfm=pfm,
        alphabet=BASES,
        consensus=record.get("consensus", ""),
        quality=record.get("quality", ""),
        datatype=record.get("datatype", ""),
    )


def load_crp():
    path = DATA_DIR / "crp_energy_matrix.txt"
    rows = []
    with path.open() as handle:
        for line in handle:
            if line.startswith("#") or line.startswith("pos") or not line.strip():
                continue
            rows.append([float(x) for x in line.split()[1:]])
    energy = np.array(rows, dtype=float)
    theta = -energy
    pfm = softmax_rows(theta, CRP_LOGO_TEMPERATURE)
    theta = standardized_weight_matrix(theta)
    return dict(
        query="CRP",
        tf="CRP",
        motif_id="CRP_Kinney2010",
        source="Kinney et al. 2010",
        theta=theta,
        pfm=pfm,
        alphabet=BASES,
        consensus="",
        quality="",
        datatype="energy",
    )


def load_abf1(key, motif_id):
    path = DATA_DIR / "abf1_mean_emats.mat"
    emat = np.array(loadmat(path)[key], dtype=float)
    theta = -emat
    pfm = softmax_rows(theta, ABF1_LOGO_TEMPERATURE)
    theta = standardized_weight_matrix(theta)
    return dict(
        query="ABF1",
        tf="ABF1",
        motif_id=motif_id,
        source="ABF1 energy matrix",
        theta=theta,
        pfm=pfm,
        alphabet=BASES,
        consensus="",
        quality="",
        datatype="energy",
    )


def load_record(query, hocomoco_records):
    local = query.upper()
    if local == "CRP":
        return load_crp()
    if local in ["ABF1", "ABF1_MUK"]:
        return load_abf1("abf1_muk_emat", "ABF1_MUK")
    if local == "ABF1_LEE":
        return load_abf1("abf1_lee_emat", "ABF1_LEE")
    return record_from_hocomoco(query, hocomoco_records)

In [ ]:
print("Define density analysis functions")

def power_model(epsilon, q, b, alpha):
    return q + b * epsilon**alpha


def safe_id(text):
    return "".join(ch if ch.isalnum() else "_" for ch in text)


def unpack_touzet_count(result):
    if isinstance(result, tuple):
        return 0.5 * (float(result[0]) + float(result[1]))
    return float(result)


def enumerate_scores(theta, max_sequences):
    L, C = theta.shape
    total = C**L
    if total > max_sequences:
        return None, total
    scores = np.array([0.0])
    for row in theta:
        scores = (scores[:, None] + row[None, :]).ravel()
    return scores, total


def exact_density(theta, landscape):
    scores, total = enumerate_scores(theta, EXACT_MAX_SEQUENCES)
    if scores is None:
        return dict(total=total, epsilon=None, log_density=None)
    score_range = landscape.F_max - landscape.F_min
    epsilon = (landscape.F_max - scores) / score_range
    bins = np.geomspace(EPSILON_LIMITS[0], EPSILON_LIMITS[1], N_EXACT_BINS + 1)
    counts, edges = np.histogram(epsilon, bins=bins)
    widths = np.diff(edges) * score_range
    density = counts / widths
    centers = np.sqrt(edges[:-1] * edges[1:])
    ok = density > 0
    return dict(total=total, epsilon=centers[ok], log_density=np.log(density[ok]))


def touzet_density(record, theta, F_edges):
    cache = TOUZET_CACHE_DIR / f"{safe_id(record['motif_id'])}_zscaled_bins{N_TOUZET_BINS}_tol{TOUZET_KW['tol']}.npz"
    if cache.exists():
        data = np.load(cache)
        F_edges = data["F_edges"]
        N_geq = data["N_geq"]
    else:
        print(f"  Computing z-scaled Touzet density for {record['motif_id']}")
        N_geq = np.array([unpack_touzet_count(tail_count(theta, F, **TOUZET_KW)) for F in F_edges])
        N_geq = np.minimum.accumulate(N_geq)
        np.savez(cache, F_edges=F_edges, N_geq=N_geq)
    widths = np.diff(F_edges)
    density = (N_geq[:-1] - N_geq[1:]) / widths
    log_density = np.full_like(density, np.nan, dtype=float)
    ok = np.isfinite(density) & (density > 0)
    log_density[ok] = np.log(density[ok])
    return dict(edges=F_edges, log_density=log_density, ok=ok)


def analyze_record(record):
    theta = np.asarray(record["theta"], dtype=float)
    L, C = theta.shape
    landscape = Landscape(theta)
    f_max = landscape.F_max
    mu = 0.0
    sigma2 = float(theta.var(axis=1).sum())
    endpoint_gap = max(1e-6, 1e-4 * max(f_max - mu, 1.0))
    F = np.linspace(mu, f_max - endpoint_gap, N_SADDLE_POINTS)
    F_touzet_edges = np.linspace(mu, f_max - endpoint_gap, N_TOUZET_BINS + 1)
    epsilon = (f_max - F) / L
    density = landscape.density(F, method="saddlepoint")
    log_sp = np.full_like(F, np.nan, dtype=float)
    ok = np.isfinite(density) & (density > 0)
    log_sp[ok] = np.log(density[ok])
    log_bulk = (
        L * np.log(C)
        - 0.5 * np.log(2 * np.pi * sigma2)
        - (F - mu) ** 2 / (2 * sigma2)
    )
    eps_max = (f_max - mu) / L
    fit_mask = ok & (epsilon < FIT_EPSILON_RANGE[1] * eps_max) & (epsilon > 1e-6)
    q, b, alpha = np.nan, np.nan, np.nan
    log_fit = np.full_like(F, np.nan, dtype=float)
    if fit_mask.sum() >= 10:
        x = epsilon[fit_mask]
        y = log_sp[fit_mask]
        p0 = [float(y[-1]), max(float(y[0] - y[-1]), 1.0), 0.5]
        bounds = ([-np.inf, 0.0, 0.05], [np.inf, np.inf, 1.2])
        q, b, alpha = curve_fit(power_model, x, y, p0=p0, bounds=bounds, maxfev=20000)[0]
        log_fit = power_model(epsilon, q, b, alpha)
    exact = exact_density(theta, landscape)
    touzet = touzet_density(record, theta, F_touzet_edges)
    return dict(
        record=record,
        landscape=landscape,
        L=L,
        C=C,
        F=F,
        f_max=f_max,
        mu=mu,
        log_sp=log_sp,
        log_bulk=log_bulk,
        log_fit=log_fit,
        ok=ok,
        fit_mask=fit_mask,
        alpha=alpha,
        exact=exact,
        touzet=touzet,
        logo_matrix=theta,
    )

In [ ]:
print("Load selected motifs and compute densities")

hocomoco_records = load_hocomoco_records(HOCOMOCO_COLLECTION)
records = [load_record(tf_name, hocomoco_records) for tf_name in TF_NAMES]
analyses = [analyze_record(record) for record in records]
analyses = sorted(analyses, key=lambda analysis: -analysis["L"])

for analysis in analyses:
    record = analysis["record"]
    exact = analysis["exact"]
    alpha = analysis["alpha"]
    alpha_text = "NA" if not np.isfinite(alpha) else f"{alpha:.3f}"
    exact_text = "saddle only" if exact["epsilon"] is None else f"enumerated {exact['total']:,} sequences"
    print(f"  {record['query']} -> {record['motif_id']}, L={analysis['L']}, C={analysis['C']}, alpha={alpha_text}, {exact_text}")

In [ ]:
print("Make figure S3")

letters = list("abcdefghijklmnopqrstuvwxyz")
n_rows = len(analyses)
fig, axes = plt.subplots(
    n_rows,
    3,
    figsize=(1.25 * sc.TEXT_WIDTH, max(2.1, 1.75 * n_rows)),
    gridspec_kw=dict(width_ratios=[1.0, 1.05, 1.05]),
)

if n_rows == 1:
    axes = np.array([axes])

for row, analysis in enumerate(analyses):
    record = analysis["record"]
    logo_ax = axes[row, 0]
    density_ax = axes[row, 1]
    scaled_ax = axes[row, 2]
    logo_matrix = analysis["logo_matrix"]
    logo_df = pd.DataFrame(logo_matrix, columns=record["alphabet"])
    logomaker.Logo(logo_df, ax=logo_ax, 
                   color_scheme="classic", 
                   font_name='Arial Rounded MT Bold', 
                   shade_below=0.5, 
                   fade_below=0.5, show_spines=False)
    y_abs = float(np.nanmax(np.abs(logo_matrix)))
    logo_ax.set_ylim(-1.12 * y_abs, 1.12 * y_abs)
    logo_ax.axhline(0, color="0.2", lw=0.5)
    logo_ax.set_ylabel("Effect")
    logo_ax.set_xlabel("position")
    logo_ax.set_title(f"({letters[row]}) {record['tf']} ($L={analysis['L']}$, $C={analysis['C']}$)", loc="left")
    step = max(1, analysis["L"] // 10)
    xticks = np.arange(0, analysis["L"], step)
    logo_ax.set_xticks(xticks)
    logo_ax.set_xticklabels(xticks + 1)

    F = analysis["F"]
    ok = analysis["ok"]
    density_ax.plot(F[ok], analysis["log_sp"][ok], **sc.STYLES["saddle"])
    touzet = analysis["touzet"]
    density_ax.stairs(
        touzet["log_density"],
        touzet["edges"],
        baseline=None,
        color="k",
        lw=0.75,
        label="Touzet",
        zorder=0.5,
    )
    density_ax.plot(F, analysis["log_bulk"], **sc.STYLES["bulk"])
    if np.isfinite(analysis["alpha"]):
        density_ax.plot(F, analysis["log_fit"], **sc.STYLES["peak"])
        F_lo, F_hi = F[ok].min(), F[ok].max()
        F_label = F_lo + 0.25 * (F_hi - F_lo)
        y_label = 0.84
        alpha_text = f"{analysis['alpha']:.2f}"
        density_ax.text(
            F_label,
            y_label,
            r"  $\alpha=" + alpha_text + r"$",
            fontsize=sc.ANNOTATION_FONTSIZE,
            color=sc.STYLES["peak"]["color"],
            va="bottom",
            ha="left",
            transform=density_ax.get_xaxis_transform(),
        )
    err_fit = np.abs(analysis["log_sp"] - analysis["log_fit"])
    err_bulk = np.abs(analysis["log_sp"] - analysis["log_bulk"])
    diff = err_fit - err_bulk
    valid = ok & np.isfinite(diff)
    F_cross = None
    for i in range(len(F) - 1):
        if valid[i] and valid[i + 1] and diff[i] > 0 and diff[i + 1] <= 0:
            w = diff[i] / (diff[i] - diff[i + 1])
            F_cross = F[i] + w * (F[i + 1] - F[i])
            break
    density_ax.axvline(analysis["mu"], **sc.STYLES["F_mean"])
    density_ax.axvline(analysis["f_max"], **sc.STYLES["F_max"])
    if F_cross is not None:
        r_val = (analysis["f_max"] - F_cross) / (analysis["f_max"] - analysis["mu"])
        density_ax.axvline(F_cross, **sc.STYLES["F_cross"])
    ymin, ymax = density_ax.get_ylim()
    density_ax.set_ylim(bottom=0, top=ymax + 0.08 * (ymax - ymin))
    density_ax.set_xlabel(r"$F$  (z-score)")
    density_ax.set_ylabel(r"$\log\,\rho(F)$")
    density_ax.set_title("")
    if row == 0:
        density_ax.legend(
            **sc.LEGEND_KW,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.05),
            ncol=3,
            columnspacing=1.0,
            handletextpad=0.5,
        )
    if F_cross is not None:
        ylim = density_ax.get_ylim()
        y_pos = ylim[0] + 0.90 * (ylim[1] - ylim[0])
        density_ax.text(
            F_cross,
            y_pos,
            f"  $r={r_val:.2f}$",
            fontsize=sc.ANNOTATION_FONTSIZE,
            color=sc.STYLES["F_cross"]["color"],
            va="center",
            ha="left",
        )

    alpha = analysis["alpha"]
    if np.isfinite(alpha):
        epsilon = (analysis["f_max"] - F) / analysis["L"]
        x = epsilon**alpha
        positive = np.isfinite(x) & (x > 0)
        scaled_ok = ok & positive & np.isfinite(analysis["log_sp"])
        scaled_ax.plot(x[scaled_ok], analysis["log_sp"][scaled_ok], **sc.STYLES["saddle"])
        scaled_ax.plot(x[positive], analysis["log_bulk"][positive], **sc.STYLES["bulk"])
        scaled_ax.plot(x[positive], analysis["log_fit"][positive], **sc.STYLES["peak"])
        touzet_x_edges = ((analysis["f_max"] - touzet["edges"]) / analysis["L"])**alpha
        scaled_ax.stairs(
            touzet["log_density"][::-1],
            touzet_x_edges[::-1],
            baseline=None,
            color="k",
            lw=0.75,
            label="Touzet",
            zorder=0.5,
        )
        eps_mu = (analysis["f_max"] - analysis["mu"]) / analysis["L"]
        if np.isfinite(eps_mu) and eps_mu > 0:
            scaled_ax.axvline(eps_mu**alpha, **sc.STYLES["F_mean"])
        if F_cross is not None:
            eps_cross = (analysis["f_max"] - F_cross) / analysis["L"]
            if np.isfinite(eps_cross) and eps_cross > 0:
                x_cross = eps_cross**alpha
                scaled_ax.axvline(x_cross, **sc.STYLES["F_cross"])
                ylim = density_ax.get_ylim()
                y_pos = ylim[0] + 0.90 * (ylim[1] - ylim[0])
                scaled_ax.text(
                    x_cross,
                    y_pos,
                    f"  $r={r_val:.2f}$",
                    fontsize=sc.ANNOTATION_FONTSIZE,
                    color=sc.STYLES["F_cross"]["color"],
                    va="center",
                    ha="left",
                )
        scaled_ax.set_xlim(left=0)
        scaled_ax.set_ylim(density_ax.get_ylim())
        scaled_ax.set_xlabel(r"$\epsilon^\alpha\quad(\alpha = " + f"{alpha:.2f}" + r")$")
        scaled_ax.set_ylabel(r"$\log\,\rho(F)$")
    else:
        scaled_ax.axis("off")

fig.tight_layout(w_pad=0.8)

if SAVE_FIGURE:
    pdf_path = FIG_DIR / "figS3.pdf"
    png_path = FIG_DIR / "figS3.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    print(f"  Saved {pdf_path}")
    print(f"  Saved {png_path}")

plt.show()
